# Perbaikan Harga Beras: Gunung Sitoli dan Pematang Siantar
Notebook ini sudah memuat data dari dua file Excel yang Ardha kirim. Jadi, **tidak perlu mengunggah file Excel lagi** ke Google Colab.

Alur kerja:
1. Membaca data Excel yang sudah ditanam di notebook.
2. Mengubah format tabel menjadi format panjang (long format).
3. Memeriksa kecocokan dengan 30 harga yang masih `NULL` di `DB_PUSAT`.
4. Memperbarui hanya baris yang harganya masih `NULL` di `DB_PUSAT`.
5. Memperbarui hanya baris `NULL` untuk **Gunung Sitoli dan Pematang Siantar** di `DW_HARGA_BERAS_SUMUT`.

**Padang Sidempuan tidak diubah sama sekali.** Data wilayah ini tetap seperti kondisi sebelumnya.

**Catatan:** Jalankan sel secara berurutan. Periksa hasil validasi sebelum menjalankan sel update database. Masukkan connection string melalui prompt rahasia, jangan tulis langsung di notebook.

> **PENTING:** Dua file Excel yang disertakan hanya untuk memperbaiki **Gunung Sitoli** dan **Pematang Siantar**. **Padang Sidempuan tetap dipertahankan apa adanya**: tidak di-update dari file Excel dan tidak dihapus.

In [4]:
import pandas as pd
import json
import re
from getpass import getpass
import psycopg2
from IPython.display import display

print("Library siap digunakan.")

Library siap digunakan.


## 1. Data Excel yang sudah disertakan

In [5]:
data_excel = {
  "GUNUNG SITOLI": {
    "columns": [
      "No",
      "Komoditas (Rp)",
      "01/ 01/ 2026",
      "02/ 02/ 2026",
      "02/ 03/ 2026",
      "01/ 04/ 2026",
      "01/ 05/ 2026",
      "01/ 06/ 2026",
      "01/ 07/ 2026",
      "03/ 08/ 2026",
      "01/ 09/ 2026",
      "01/ 10/ 2026",
      "02/ 11/ 2026"
    ],
    "rows": [
      {
        "No": "I",
        "Komoditas (Rp)": "Beras",
        "01/ 01/ 2026": "13,850",
        "02/ 02/ 2026": "13,850",
        "02/ 03/ 2026": "13,950",
        "01/ 04/ 2026": "13,950",
        "01/ 05/ 2026": "14.024999999999999",
        "01/ 06/ 2026": "14,100",
        "01/ 07/ 2026": "14,200",
        "03/ 08/ 2026": "14,200",
        "01/ 09/ 2026": "14,200",
        "01/ 10/ 2026": "14,200",
        "02/ 11/ 2026": "14,200"
      },
      {
        "No": "1",
        "Komoditas (Rp)": "Beras Kualitas Bawah I",
        "01/ 01/ 2026": "13,900",
        "02/ 02/ 2026": "13,900",
        "02/ 03/ 2026": "13,900",
        "01/ 04/ 2026": "13,900",
        "01/ 05/ 2026": "13.9",
        "01/ 06/ 2026": "13,900",
        "01/ 07/ 2026": "13,900",
        "03/ 08/ 2026": "13,900",
        "01/ 09/ 2026": "13,900",
        "01/ 10/ 2026": "13,900",
        "02/ 11/ 2026": "13,900"
      },
      {
        "No": "2",
        "Komoditas (Rp)": "Beras Kualitas Bawah II",
        "01/ 01/ 2026": "14,500",
        "02/ 02/ 2026": "14,500",
        "02/ 03/ 2026": "14,500",
        "01/ 04/ 2026": "14,500",
        "01/ 05/ 2026": "14.5",
        "01/ 06/ 2026": "14,500",
        "01/ 07/ 2026": "14,500",
        "03/ 08/ 2026": "14,500",
        "01/ 09/ 2026": "14,500",
        "01/ 10/ 2026": "14,500",
        "02/ 11/ 2026": "14,500"
      },
      {
        "No": "3",
        "Komoditas (Rp)": "Beras Kualitas Medium I",
        "01/ 01/ 2026": "14,500",
        "02/ 02/ 2026": "14,500",
        "02/ 03/ 2026": "14,500",
        "01/ 04/ 2026": "14,500",
        "01/ 05/ 2026": "14.5",
        "01/ 06/ 2026": "14,500",
        "01/ 07/ 2026": "15,000",
        "03/ 08/ 2026": "15,000",
        "01/ 09/ 2026": "15,000",
        "01/ 10/ 2026": "15,000",
        "02/ 11/ 2026": "15,000"
      },
      {
        "No": "4",
        "Komoditas (Rp)": "Beras Kualitas Medium II",
        "01/ 01/ 2026": "13,400",
        "02/ 02/ 2026": "13,400",
        "02/ 03/ 2026": "13,400",
        "01/ 04/ 2026": "13,400",
        "01/ 05/ 2026": "13.65",
        "01/ 06/ 2026": "13,900",
        "01/ 07/ 2026": "13,900",
        "03/ 08/ 2026": "13,900",
        "01/ 09/ 2026": "13,900",
        "01/ 10/ 2026": "13,900",
        "02/ 11/ 2026": "13,900"
      },
      {
        "No": "5",
        "Komoditas (Rp)": "Beras Kualitas Super I",
        "01/ 01/ 2026": "13,400",
        "02/ 02/ 2026": "13,400",
        "02/ 03/ 2026": "13,400",
        "01/ 04/ 2026": "13,400",
        "01/ 05/ 2026": "13.65",
        "01/ 06/ 2026": "13,900",
        "01/ 07/ 2026": "13,900",
        "03/ 08/ 2026": "13,900",
        "01/ 09/ 2026": "13,900",
        "01/ 10/ 2026": "13,900",
        "02/ 11/ 2026": "13,900"
      },
      {
        "No": "6",
        "Komoditas (Rp)": "Beras Kualitas Super II",
        "01/ 01/ 2026": "13,400",
        "02/ 02/ 2026": "13,400",
        "02/ 03/ 2026": "13,900",
        "01/ 04/ 2026": "13,900",
        "01/ 05/ 2026": "13.9",
        "01/ 06/ 2026": "13,900",
        "01/ 07/ 2026": "13,900",
        "03/ 08/ 2026": "13,900",
        "01/ 09/ 2026": "13,900",
        "01/ 10/ 2026": "13,900",
        "02/ 11/ 2026": "13,900"
      }
    ]
  },
  "PEMATANG SIANTAR": {
    "columns": [
      "No",
      "Komoditas (Rp)",
      "01/ 01/ 2026",
      "02/ 02/ 2026",
      "02/ 03/ 2026",
      "01/ 04/ 2026",
      "01/ 05/ 2026",
      "01/ 06/ 2026",
      "01/ 07/ 2026",
      "03/ 08/ 2026",
      "01/ 09/ 2026",
      "01/ 10/ 2026",
      "02/ 11/ 2026"
    ],
    "rows": [
      {
        "No": "I",
        "Komoditas (Rp)": "Beras",
        "01/ 01/ 2026": "13,250",
        "02/ 02/ 2026": "13,250",
        "02/ 03/ 2026": "13,250",
        "01/ 04/ 2026": "13,250",
        "01/ 05/ 2026": "13.425",
        "01/ 06/ 2026": "13,600",
        "01/ 07/ 2026": "13,050",
        "03/ 08/ 2026": "13.8",
        "01/ 09/ 2026": "14,550",
        "01/ 10/ 2026": "14,550",
        "02/ 11/ 2026": "14,550"
      },
      {
        "No": "1",
        "Komoditas (Rp)": "Beras Kualitas Bawah I",
        "01/ 01/ 2026": "11,000",
        "02/ 02/ 2026": "11,000",
        "02/ 03/ 2026": "11,000",
        "01/ 04/ 2026": "11,000",
        "01/ 05/ 2026": "11.4",
        "01/ 06/ 2026": "11,800",
        "01/ 07/ 2026": "11,800",
        "03/ 08/ 2026": "12.2",
        "01/ 09/ 2026": "12,600",
        "01/ 10/ 2026": "12,600",
        "02/ 11/ 2026": "12,600"
      },
      {
        "No": "2",
        "Komoditas (Rp)": "Beras Kualitas Bawah II",
        "01/ 01/ 2026": "12,800",
        "02/ 02/ 2026": "12,800",
        "02/ 03/ 2026": "12,800",
        "01/ 04/ 2026": "12,800",
        "01/ 05/ 2026": "12.9",
        "01/ 06/ 2026": "13,000",
        "01/ 07/ 2026": "13,000",
        "03/ 08/ 2026": "13.25",
        "01/ 09/ 2026": "13,500",
        "01/ 10/ 2026": "13,500",
        "02/ 11/ 2026": "13,500"
      },
      {
        "No": "3",
        "Komoditas (Rp)": "Beras Kualitas Medium I",
        "01/ 01/ 2026": "13,100",
        "02/ 02/ 2026": "13,100",
        "02/ 03/ 2026": "13,100",
        "01/ 04/ 2026": "13,100",
        "01/ 05/ 2026": "13.225",
        "01/ 06/ 2026": "13,350",
        "01/ 07/ 2026": "13,350",
        "03/ 08/ 2026": "13.825",
        "01/ 09/ 2026": "14,300",
        "01/ 10/ 2026": "14,300",
        "02/ 11/ 2026": "14,300"
      },
      {
        "No": "4",
        "Komoditas (Rp)": "Beras Kualitas Medium II",
        "01/ 01/ 2026": "13,850",
        "02/ 02/ 2026": "13,850",
        "02/ 03/ 2026": "13,850",
        "01/ 04/ 2026": "13,850",
        "01/ 05/ 2026": "13.975",
        "01/ 06/ 2026": "14,100",
        "01/ 07/ 2026": "14,100",
        "03/ 08/ 2026": "14.649999999999999",
        "01/ 09/ 2026": "15,200",
        "01/ 10/ 2026": "15,200",
        "02/ 11/ 2026": "15,200"
      },
      {
        "No": "5",
        "Komoditas (Rp)": "Beras Kualitas Super I",
        "01/ 01/ 2026": "14,000",
        "02/ 02/ 2026": "14,000",
        "02/ 03/ 2026": "14,000",
        "01/ 04/ 2026": "14,000",
        "01/ 05/ 2026": "14.15",
        "01/ 06/ 2026": "14,300",
        "01/ 07/ 2026": "14.591836734693878",
        "03/ 08/ 2026": "15.145918367346939",
        "01/ 09/ 2026": "15,700",
        "01/ 10/ 2026": "15,700",
        "02/ 11/ 2026": "15,700"
      },
      {
        "No": "6",
        "Komoditas (Rp)": "Beras Kualitas Super II",
        "01/ 01/ 2026": "14,850",
        "02/ 02/ 2026": "14,850",
        "02/ 03/ 2026": "14,850",
        "01/ 04/ 2026": "14,850",
        "01/ 05/ 2026": "14.975",
        "01/ 06/ 2026": "15,100",
        "01/ 07/ 2026": "15.252525252525253",
        "03/ 08/ 2026": "15.576262626262626",
        "01/ 09/ 2026": "15,900",
        "01/ 10/ 2026": "15,900",
        "02/ 11/ 2026": "15,900"
      }
    ]
  }
}

df_gunung = pd.DataFrame(
    data_excel["GUNUNG SITOLI"]["rows"],
    columns=data_excel["GUNUNG SITOLI"]["columns"]
)
df_pematang = pd.DataFrame(
    data_excel["PEMATANG SIANTAR"]["rows"],
    columns=data_excel["PEMATANG SIANTAR"]["columns"]
)

print("Gunung Sitoli:", df_gunung.shape)
print("Pematang Siantar:", df_pematang.shape)
display(df_gunung)
display(df_pematang)

Gunung Sitoli: (7, 13)
Pematang Siantar: (7, 13)


,No,Komoditas (Rp),01/ 01/ 2026,02/ 02/ 2026,02/ 03/ 2026,01/ 04/ 2026,01/ 05/ 2026,01/ 06/ 2026,01/ 07/ 2026,03/ 08/ 2026,01/ 09/ 2026,01/ 10/ 2026,02/ 11/ 2026
0,I,Beras,"13,850","13,850","13,950","13,950",14.024999999999999,"14,100","14,200","14,200","14,200","14,200","14,200"
1,1,Beras Kualitas Bawah I,"13,900","13,900","13,900","13,900",13.9,"13,900","13,900","13,900","13,900","13,900","13,900"
2,2,Beras Kualitas Bawah II,"14,500","14,500","14,500","14,500",14.5,"14,500","14,500","14,500","14,500","14,500","14,500"
3,3,Beras Kualitas Medium I,"14,500","14,500","14,500","14,500",14.5,"14,500","15,000","15,000","15,000","15,000","15,000"
4,4,Beras Kualitas Medium II,"13,400","13,400","13,400","13,400",13.65,"13,900","13,900","13,900","13,900","13,900","13,900"
5,5,Beras Kualitas Super I,"13,400","13,400","13,400","13,400",13.65,"13,900","13,900","13,900","13,900","13,900","13,900"
6,6,Beras Kualitas Super II,"13,400","13,400","13,900","13,900",13.9,"13,900","13,900","13,900","13,900","13,900","13,900"


,No,Komoditas (Rp),01/ 01/ 2026,02/ 02/ 2026,02/ 03/ 2026,01/ 04/ 2026,01/ 05/ 2026,01/ 06/ 2026,01/ 07/ 2026,03/ 08/ 2026,01/ 09/ 2026,01/ 10/ 2026,02/ 11/ 2026
0,I,Beras,"13,250","13,250","13,250","13,250",13.425,"13,600","13,050",13.8,"14,550","14,550","14,550"
1,1,Beras Kualitas Bawah I,"11,000","11,000","11,000","11,000",11.4,"11,800","11,800",12.2,"12,600","12,600","12,600"
2,2,Beras Kualitas Bawah II,"12,800","12,800","12,800","12,800",12.9,"13,000","13,000",13.25,"13,500","13,500","13,500"
3,3,Beras Kualitas Medium I,"13,100","13,100","13,100","13,100",13.225,"13,350","13,350",13.825,"14,300","14,300","14,300"
4,4,Beras Kualitas Medium II,"13,850","13,850","13,850","13,850",13.975,"14,100","14,100",14.649999999999999,"15,200","15,200","15,200"
5,5,Beras Kualitas Super I,"14,000","14,000","14,000","14,000",14.15,"14,300",14.591836734693878,15.145918367346939,"15,700","15,700","15,700"
6,6,Beras Kualitas Super II,"14,850","14,850","14,850","14,850",14.975,"15,100",15.252525252525253,15.576262626262626,"15,900","15,900","15,900"


## 2. Transformasi data
Di file sumber, angka seperti `13,850` berarti Rp13.850, sedangkan angka seperti `14.025` berarti Rp14.025.000? **Tidak**: dalam file ini angka bertitik merupakan nilai ribuan desimal, sehingga `14.025` dikonversi menjadi `14.025 × 1.000 = 14.025` rupiah (nilai numerik 14025).

In [6]:
def konversi_harga(nilai):
    if nilai is None or str(nilai).strip() == "":
        return None

    teks = str(nilai).strip()

    # Format koma sebagai pemisah ribuan, contoh: 13,850 -> 13850
    if "," in teks:
        return float(teks.replace(",", ""))

    # Format titik sebagai desimal dalam satuan ribuan, contoh: 14.025 -> 14025
    return round(float(teks) * 1000, 3)


def ubah_ke_long(df, wilayah):
    hasil = []
    kolom_tanggal = df.columns[2:]  # mulai dari kolom tanggal

    for _, row in df.iterrows():
        kualitas = str(row["Komoditas (Rp)"]).strip()

        for kolom in kolom_tanggal:
            tanggal_teks = re.sub(r"\s+", "", str(kolom)).replace("/", "-")
            tanggal = pd.to_datetime(
                tanggal_teks, format="%d-%m-%Y", errors="raise"
            ).date()

            hasil.append({
                "wilayah_sumber": wilayah,
                "kualitas_beras": kualitas,
                "tanggal": tanggal,
                "harga": konversi_harga(row[kolom])
            })

    return pd.DataFrame(hasil)


df_baru = pd.concat([
    ubah_ke_long(df_gunung, "GUNUNG SITOLI"),
    ubah_ke_long(df_pematang, "PEMATANG SIANTAR")
], ignore_index=True)

print("Jumlah baris hasil transformasi:", len(df_baru))
print("Jumlah harga kosong dalam file:", df_baru["harga"].isna().sum())
print("Jumlah kombinasi duplikat:", df_baru.duplicated(
    ["wilayah_sumber", "kualitas_beras", "tanggal"]
).sum())
display(df_baru.head(15))

Jumlah baris hasil transformasi: 154
Jumlah harga kosong dalam file: 0
Jumlah kombinasi duplikat: 0


,wilayah_sumber,kualitas_beras,tanggal,harga
0,GUNUNG SITOLI,Beras,2026-01-01,13850.0
1,GUNUNG SITOLI,Beras,2026-02-02,13850.0
2,GUNUNG SITOLI,Beras,2026-03-02,13950.0
3,GUNUNG SITOLI,Beras,2026-04-01,13950.0
4,GUNUNG SITOLI,Beras,2026-05-01,14025.0
5,GUNUNG SITOLI,Beras,2026-06-01,14100.0
6,GUNUNG SITOLI,Beras,2026-07-01,14200.0
7,GUNUNG SITOLI,Beras,2026-08-03,14200.0
8,GUNUNG SITOLI,Beras,2026-09-01,14200.0
9,GUNUNG SITOLI,Beras,2026-10-01,14200.0


## 3. Koneksi ke database
Masukkan connection string `DB_PUSAT` dan `DW_HARGA_BERAS_SUMUT` saat diminta. Keduanya tidak akan ditampilkan di output.

In [7]:
SOURCE_URL = getpass("Connection string DB_PUSAT: ")
DW_URL = getpass("Connection string DW_HARGA_BERAS_SUMUT: ")

conn_source = psycopg2.connect(SOURCE_URL)
conn_dw = psycopg2.connect(DW_URL)

print("Koneksi DB_PUSAT dan DW_HARGA_BERAS_SUMUT berhasil.")

Connection string DB_PUSAT: ··········
Connection string DW_HARGA_BERAS_SUMUT: ··········
Koneksi DB_PUSAT dan DW_HARGA_BERAS_SUMUT berhasil.


## 4. Cocokkan harga NULL untuk dua wilayah yang diperbaiki di DB_PUSAT
Sel ini hanya membaca database, belum mengubah data.

In [8]:
query_null_source = """
SELECT kode_harga, wilayah_sumber, kualitas_beras, tanggal
FROM public.harga_beras_pusat
WHERE harga IS NULL
  AND wilayah_sumber IN ('GUNUNG SITOLI', 'PEMATANG SIANTAR')
ORDER BY tanggal, wilayah_sumber, kualitas_beras;
"""

df_null_source = pd.read_sql_query(query_null_source, conn_source)
df_null_source["tanggal"] = pd.to_datetime(df_null_source["tanggal"]).dt.date

df_update = df_null_source.merge(
    df_baru,
    on=["wilayah_sumber", "kualitas_beras", "tanggal"],
    how="left",
    validate="one_to_one"
)

print("Jumlah harga NULL di DB_PUSAT:", len(df_null_source))
print("Jumlah yang berhasil ditemukan di Excel:", df_update["harga"].notna().sum())
print("Jumlah yang belum ditemukan:", df_update["harga"].isna().sum())
display(df_update)

/tmp/ipykernel_1996/2112238673.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_null_source = pd.read_sql_query(query_null_source, conn_source)


Jumlah harga NULL di DB_PUSAT: 30
Jumlah yang berhasil ditemukan di Excel: 30
Jumlah yang belum ditemukan: 0


,kode_harga,wilayah_sumber,kualitas_beras,tanggal,harga
0,GNS-005,GUNUNG SITOLI,Beras,2026-05-01,14025.000
1,GNS-016,GUNUNG SITOLI,Beras Kualitas Bawah I,2026-05-01,13900.000
2,GNS-027,GUNUNG SITOLI,Beras Kualitas Bawah II,2026-05-01,14500.000
3,GNS-038,GUNUNG SITOLI,Beras Kualitas Medium I,2026-05-01,14500.000
4,GNS-049,GUNUNG SITOLI,Beras Kualitas Medium II,2026-05-01,13650.000
5,GNS-060,GUNUNG SITOLI,Beras Kualitas Super I,2026-05-01,13650.000
6,GNS-071,GUNUNG SITOLI,Beras Kualitas Super II,2026-05-01,13900.000
7,PSI-005,PEMATANG SIANTAR,Beras,2026-05-01,13425.000
8,PSI-016,PEMATANG SIANTAR,Beras Kualitas Bawah I,2026-05-01,11400.000
9,PSI-027,PEMATANG SIANTAR,Beras Kualitas Bawah II,2026-05-01,12900.000


**Checkpoint:** hasil yang diharapkan adalah 30 baris NULL, 30 harga ditemukan, dan 0 harga belum ditemukan. Jika hasilnya berbeda, jangan jalankan sel update berikutnya dulu.

## 5. Update DB_PUSAT
Sel ini hanya mengisi `harga` pada baris `NULL` milik **Gunung Sitoli dan Pematang Siantar**. Data Padang Sidempuan tidak disentuh, dan harga yang sudah terisi tidak ditimpa.

In [9]:
if (len(df_update) == 30
    and set(df_update["wilayah_sumber"].unique()) == {"GUNUNG SITOLI", "PEMATANG SIANTAR"}
    and df_update["harga"].notna().all()):
    cur_source = conn_source.cursor()
    sql_update_source = """
    UPDATE public.harga_beras_pusat
    SET harga = %s
    WHERE kode_harga = %s
      AND wilayah_sumber IN ('GUNUNG SITOLI', 'PEMATANG SIANTAR')
      AND harga IS NULL;
    """

    values_source = [
        (float(row.harga), row.kode_harga)
        for row in df_update.itertuples(index=False)
    ]

    cur_source.executemany(sql_update_source, values_source)
    print("Baris DB_PUSAT yang diperbarui:", cur_source.rowcount)
    conn_source.commit()
    cur_source.close()
else:
    raise ValueError("Validasi belum sesuai. Update DB_PUSAT dibatalkan.")

Baris DB_PUSAT yang diperbarui: 30


## 6. Verifikasi DB_PUSAT
Pastikan tidak ada lagi harga NULL untuk dua wilayah yang diperbaiki.

In [10]:
query_check_source = """
SELECT wilayah_sumber,
       COUNT(*) AS total_data,
       COUNT(*) FILTER (WHERE harga IS NULL) AS harga_null
FROM public.harga_beras_pusat
WHERE wilayah_sumber IN ('GUNUNG SITOLI', 'PEMATANG SIANTAR')
GROUP BY wilayah_sumber
ORDER BY wilayah_sumber;
"""

df_check_source = pd.read_sql_query(query_check_source, conn_source)
display(df_check_source)

/tmp/ipykernel_1996/1112954778.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_check_source = pd.read_sql_query(query_check_source, conn_source)


,wilayah_sumber,total_data,harga_null
0,GUNUNG SITOLI,77,0
1,PEMATANG SIANTAR,77,0


## 7. Update DW_HARGA_BERAS_SUMUT
Sel ini hanya mengisi harga `NULL` untuk **Gunung Sitoli dan Pematang Siantar**. Data Padang Sidempuan tetap tidak berubah.

In [11]:
# Ambil ulang baris NULL dari fact table di data warehouse
query_null_dw = """
SELECT
    f.fact_key,
    w.nama_wilayah AS wilayah_sumber,
    k.kualitas_beras,
    t.tanggal
FROM dw.fact_harga_beras f
JOIN dw.dim_wilayah w
    ON f.wilayah_key = w.wilayah_key
JOIN dw.dim_kualitas_beras k
    ON f.kualitas_key = k.kualitas_key
JOIN dw.dim_tanggal t
    ON f.tanggal_key = t.tanggal_key
WHERE f.harga IS NULL
  AND w.nama_wilayah IN ('GUNUNG SITOLI', 'PEMATANG SIANTAR');
"""

df_null_dw = pd.read_sql_query(query_null_dw, conn_dw)
df_null_dw["tanggal"] = pd.to_datetime(df_null_dw["tanggal"]).dt.date

df_update_dw = df_null_dw.merge(
    df_baru,
    on=["wilayah_sumber", "kualitas_beras", "tanggal"],
    how="left",
    validate="one_to_one"
)

print("Harga NULL di DW:", len(df_null_dw))
print("Harga ditemukan:", df_update_dw["harga"].notna().sum())
print("Harga belum ditemukan:", df_update_dw["harga"].isna().sum())
display(df_update_dw)

/tmp/ipykernel_1996/191656663.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_null_dw = pd.read_sql_query(query_null_dw, conn_dw)


Harga NULL di DW: 0
Harga ditemukan: 0
Harga belum ditemukan: 0


,fact_key,wilayah_sumber,kualitas_beras,tanggal,harga


**Checkpoint:** pastikan semua baris NULL di DW memiliki pasangan harga dari Excel sebelum menjalankan update.

In [12]:
if df_update_dw["harga"].notna().all():
    cur_dw = conn_dw.cursor()
    sql_update_dw = """
    UPDATE dw.fact_harga_beras
    SET harga = %s,
        loaded_at = NOW()
    WHERE fact_key = %s
      AND harga IS NULL;
    """

    values_dw = [
        (float(row.harga), int(row.fact_key))
        for row in df_update_dw.itertuples(index=False)
    ]

    cur_dw.executemany(sql_update_dw, values_dw)
    print("Baris DW yang diperbarui:", cur_dw.rowcount)
    conn_dw.commit()
    cur_dw.close()
else:
    raise ValueError("Ada harga yang belum ditemukan. Update DW dibatalkan.")

Baris DW yang diperbarui: 0


## 8. Verifikasi akhir

In [13]:
query_final = """
SELECT
    w.nama_wilayah,
    COUNT(*) AS total_data,
    COUNT(*) FILTER (WHERE f.harga IS NULL) AS harga_null,
    COUNT(*) FILTER (WHERE f.harga IS NOT NULL) AS harga_terisi
FROM dw.fact_harga_beras f
JOIN dw.dim_wilayah w
    ON f.wilayah_key = w.wilayah_key
GROUP BY w.nama_wilayah
ORDER BY w.nama_wilayah;
"""

df_final = pd.read_sql_query(query_final, conn_dw)
display(df_final)

conn_source.close()
conn_dw.close()
print("Koneksi database ditutup.")

/tmp/ipykernel_1996/929574297.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_final = pd.read_sql_query(query_final, conn_dw)


,nama_wilayah,total_data,harga_null,harga_terisi


Koneksi database ditutup.


In [14]:
import pandas as pd
import json
import re
from getpass import getpass
import psycopg2
from IPython.display import display

print("Library siap digunakan.")

Library siap digunakan.
